# Mage ML pipeline: assignment version (Part C)
Based on the class notebook **Maze_ML_PipeLine**. The Mage project `iris_project/` is already in this repository,
so this notebook clones the repo, installs Mage, and runs C1–C4. Run top to bottom in Google Colab.

## 0. Get the repository and install Mage

In [1]:
REPO_URL = "https://github.com/DeepakDSA/freshcart-pipelines.git"   # <- your repo
!git clone -q {REPO_URL} /content/freshcart-pipelines
%cd /content/freshcart-pipelines
!ls

/content/freshcart-pipelines
elt	      mage_iris_assignment.ipynb  README.md	    stream
iris_project  pytest.ini		  requirements.txt  tests


In [2]:
!pip install -q uv
!uv venv -q -p 3.12 /content/mage_env
!uv pip install -q --python /content/mage_env mage-ai scikit-learn

import os
os.environ['PATH'] = '/content/mage_env/bin:' + os.environ['PATH']
!mage --help | head -5

                                                                                
 Usage: mage [OPTIONS] COMMAND [ARGS]...                                        
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion          Install completion for the current shell.      │


## C1. The pipeline and a run with tests
Final graph (after C3 and C4): `load_iris → prepare_data → {train_model → {save_results, evaluate_model}, train_knn}`.

In [3]:
!cat iris_project/pipelines/iris_classifier/metadata.yaml

blocks:
- name: load_iris
  uuid: load_iris
  type: data_loader
  language: python
  upstream_blocks: []
  downstream_blocks: [prepare_data]
- name: prepare_data
  uuid: prepare_data
  type: transformer
  language: python
  upstream_blocks: [load_iris]
  downstream_blocks: [train_model, train_knn]
- name: train_model
  uuid: train_model
  type: transformer
  language: python
  upstream_blocks: [prepare_data]
  downstream_blocks: [save_results, evaluate_model]
- name: save_results
  uuid: save_results
  type: data_exporter
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []
- name: evaluate_model
  uuid: evaluate_model
  type: transformer
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []
- name: train_knn
  uuid: train_knn
  type: transformer
  language: python
  upstream_blocks: [prepare_data]
  downstream_blocks: []
name: iris_classifier
uuid: iris_classifier
type: python
variables:
  max_depth: 3


In [4]:
!mage run iris_project iris_classifier --test 2>&1 | grep -v traitlets

INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
DBT library not installed.
Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
KNN n_neighbors = 5 | Test accuracy = 1.0
max_depth = 3 | Test accuracy = 1.0
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Saved 30 predictions to iris_predictions.csv
Confusion matrix:
predicted   0  1   2
actual              
0          10  0   0
1           0  9   0
2           0  0  11
Metrics:
   metric  value
 accuracy    1.0
precision    1.0
   recall    1.0
 f1_score    1.0
Pipeline run completed.


## C2. Tests as a gate
For each max_depth: delete `iris_predictions.csv`, run with `--runtime-vars`, record accuracy, whether
`test_accuracy` passed, and whether the CSV was created.

In [5]:
import subprocess, re, json, pandas as pd

rows = []
for depth in [1, 2, 3, 4, 5]:
    if os.path.exists('iris_predictions.csv'):
        os.remove('iris_predictions.csv')
    out = subprocess.run(['mage', 'run', 'iris_project', 'iris_classifier', '--test',
                          '--runtime-vars', json.dumps({'max_depth': depth})],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout
    acc = re.search(r'max_depth = \d+ \| Test accuracy = ([0-9.]+)', out).group(1)
    print(f'----- max_depth = {depth} -----')
    print('\n'.join(l for l in out.splitlines() if 'accuracy' in l or 'FAIL' in l or 'Saved' in l or 'tests passed' in l))
    rows.append({'max_depth': depth, 'test_accuracy': float(acc),
                 'test_accuracy passed?': 'no' if 'FAIL: test_accuracy' in out else 'yes',
                 'iris_predictions.csv created?': 'yes' if os.path.exists('iris_predictions.csv') else 'no'})

pd.DataFrame(rows)

----- max_depth = 1 -----
1/1 tests passed.
1/1 tests passed.
max_depth = 1 | Test accuracy = 0.633
FAIL: test_accuracy (block: train_model)
  File "<string>", line 42, in test_accuracy
AssertionError: Model accuracy is below 80%
0/1 tests passed.
KNN n_neighbors = 5 | Test accuracy = 1.0
----- max_depth = 2 -----
1/1 tests passed.
1/1 tests passed.
max_depth = 2 | Test accuracy = 0.967
1/1 tests passed.
Saved 30 predictions to iris_predictions.csv
 accuracy  0.967
KNN n_neighbors = 5 | Test accuracy = 1.0
1/1 tests passed.
----- max_depth = 3 -----
1/1 tests passed.
1/1 tests passed.
max_depth = 3 | Test accuracy = 1.0
1/1 tests passed.
KNN n_neighbors = 5 | Test accuracy = 1.0
Saved 30 predictions to iris_predictions.csv
 accuracy    1.0
1/1 tests passed.
----- max_depth = 4 -----
1/1 tests passed.
1/1 tests passed.
max_depth = 4 | Test accuracy = 1.0
1/1 tests passed.
Saved 30 predictions to iris_predictions.csv
 accuracy    1.0
KNN n_neighbors = 5 | Test accuracy = 1.0
1/1 tests pa

,max_depth,test_accuracy,test_accuracy passed?,iris_predictions.csv created?
0,1,0.633,no,no
1,2,0.967,yes,yes
2,3,1.000,yes,yes
3,4,1.000,yes,yes
4,5,1.000,yes,yes


Full log of the failing run (max_depth = 1), to show the test stopping the pipeline:

In [6]:
!rm -f iris_predictions.csv
!mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 1}' 2>&1 | grep -v traitlets | head -25
print('iris_predictions.csv exists:', os.path.exists('iris_predictions.csv'))

INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
DBT library not installed.
Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
KNN n_neighbors = 5 | Test accuracy = 1.0
max_depth = 1 | Test accuracy = 0.633
FAIL: test_accuracy (block: train_model)
--------------------------------------------------------------
Traceback (most recent call last):
  File "/content/mage_env/lib/python3.12/site-packages/mage_ai/data_preparation/executors/pipeline_executor.py", line 82, in execute
    loop = asyncio.get_running_loop()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: no running event loop

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/mage_env/lib/python3.12/site-packages/mage_ai/data_preparation/m

## C3. Evaluation branch: confusion matrix and metrics (max_depth = 3)

In [7]:
!cat iris_project/transformers/evaluate_model.py

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer


@transformer
def evaluate(df, *args, **kwargs):
    y_true = df['target']
    y_pred = df['predicted']

    matrix = pd.crosstab(y_true, y_pred, rownames=['actual'], colnames=['predicted'])
    print('Confusion matrix:\n' + matrix.to_string())

    metrics = pd.DataFrame({
        'metric': ['accuracy', 'precision', 'recall', 'f1_score'],
        'value': [
            round(accuracy_score(y_true, y_pred), 3),
            round(precision_score(y_true, y_pred, average='macro'), 3),
            round(recall_score(y_true, y_pred, average='macro'), 3),
            round(f1_score(y_true, y_pred, average='macro'), 3),
        ]
    })
    print('Metrics:\n' + metrics.to_string(index=False))   # added so the table appears in the run log
    return metrics


In [8]:
!mage run iris_project iris_classifier --test 2>&1 | grep -v traitlets

INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
DBT library not installed.
Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
max_depth = 3 | Test accuracy = 1.0
--------------------------------------------------------------
1/1 tests passed.
Saved 30 predictions to iris_predictions.csv
Confusion matrix:
predicted   0  1   2
actual              
0          10  0   0
1           0  9   0
2           0  0  11
KNN n_neighbors = 5 | Test accuracy = 1.0
Metrics:
   metric  value
 accuracy    1.0
precision    1.0
   recall    1.0
 f1_score    1.0
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


Same branch with max_depth = 2, where the matrix is more interesting (one mistake):

In [9]:
!mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 2}' 2>&1 | sed -n '/Confusion/,/f1_score/p'

Confusion matrix:
predicted   0  1   2
actual              
0          10  0   0
1           0  8   1
2           0  0  11
Metrics:
   metric  value
 accuracy  0.967
precision  0.972
   recall  0.963
 f1_score  0.966


## C4. Own block: a KNN classifier as a parallel branch with its own @test

In [10]:
!cat iris_project/transformers/train_knn.py

from sklearn.neighbors import KNeighborsClassifier

# Import Mage's decorators only if they aren't already loaded
if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

FEATURES = ['sepal length (cm)', 'sepal width (cm)',
            'petal length (cm)', 'petal width (cm)']


# A second model, running as a parallel branch next to train_model.
# It gets the same 150-row table (with the 'split' column) from prepare_data,
# so both models are trained and tested on exactly the same rows.
@transformer
def train_knn(df, *args, **kwargs):
    k = int(kwargs.get('n_neighbors', 5))

    train_df = df[df['split'] == 'train']
    test_df = df[df['split'] == 'test'].copy()

    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(train_df[FEATURES], train_df['target'])

    test_df['predicted'] = model.predict(test_df[FEATURES])
    accuracy = (test_df['predicted']

In [11]:
!mage run iris_project iris_classifier --test 2>&1 | grep -E 'accuracy|tests passed|completed'

1/1 tests passed.
1/1 tests passed.
max_depth = 3 | Test accuracy = 1.0
1/1 tests passed.
 accuracy    1.0
KNN n_neighbors = 5 | Test accuracy = 1.0
1/1 tests passed.
Pipeline run completed.


CI check, the same command GitHub Actions runs:

In [12]:
!uv pip install -q --python /content/mage_env pytest && /content/mage_env/bin/python -m pytest -v

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-9.2.0, pluggy-1.7.0 -- /content/mage_env/bin/python
cachedir: .pytest_cache
rootdir: /content/freshcart-pipelines
configfile: pytest.ini
testpaths: tests
plugins: anyio-4.14.2, platformdirs-4.14.5, Faker-4.14.0
collected 1 item                                                               

tests/test_prepare_data.py::test_split_is_80_20 PASSED                   [100%]

============================== 1 passed in 1.55s ===============================


## Mage UI (screenshot for C3): start the server and open it through a Cloudflare tunnel

In [13]:
import subprocess, time, urllib.request
mage_server = subprocess.Popen(['/content/mage_env/bin/mage', 'start', 'iris_project'],
                               stdout=open('mage.log', 'w'), stderr=subprocess.STDOUT)
for i in range(60):
    try:
        urllib.request.urlopen('http://localhost:6789'); print('Mage is running'); break
    except Exception:
        time.sleep(2)
else:
    print('Mage did not start - check mage.log')

Mage is running


In [14]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:6789'],
                          stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(10)
link = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
print('Open Mage here:', link[0] if link else 'Not ready yet - run this cell again')
print('Then: Pipelines -> iris_classifier -> Edit, and screenshot the graph showing the branch.')

Open Mage here: https://agrees-agreement-sing-structured.trycloudflare.com
Then: Pipelines -> iris_classifier -> Edit, and screenshot the graph showing the branch.
